# DeepPEF Evolution v4 - Colab Pro (account 2) + HuggingFace data

**Setup (no edits needed):**
- Code: GitHub `nissimbrami/DeepEF-Thesis` (public)
- Data: HuggingFace `nissimb/deepef-megascale` (public) downloaded to Colab VM disk (~50 GB, NOT Drive)
- Outputs (logs every epoch, models, state.json): account-2 Drive `MyDrive/DeepPEF_evolution_v4/`

**GPU:** L4 (Runtime -> Change runtime type -> L4).

**Rungs (evolution with auto-fallback):**

| Rung | Change | Expected PCC | Time (L4) |
|---|---|---|---|
| 0 | Baseline single-seed sanity | 0.50-0.53 | ~40 min |
| 1 | + 5-seed ensemble | 0.55-0.57 | ~3 hr |
| 2 | + serial_fusion | 0.55-0.60 | ~3 hr |
| 3 | + use_learned_aa (stacks on rung 2 if it won) | 0.55-0.61 | ~3 hr |
| 4 | Combine winners | 0.56-0.62 | ~3 hr |

**Every seed:** per-epoch PCC + best PCC + any error -> appended to `state.json` on Drive. Logs and model checkpoint also pushed to Drive immediately. Fallback verdict and error message written to history.

Just **Runtime -> Run all**. Sign in with account 2 when Cell 1 asks.

In [ ]:
# Cell 1: Mount account-2 Drive (for OUTPUTS only - logs, models, state)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
OUT_ROOT = '/content/drive/MyDrive/DeepPEF_evolution_v4'
os.makedirs(f'{OUT_ROOT}/logs',    exist_ok=True)
os.makedirs(f'{OUT_ROOT}/models',  exist_ok=True)
os.makedirs(f'{OUT_ROOT}/results', exist_ok=True)
print('Outputs will go to:', OUT_ROOT)
!df -h /content/drive 2>/dev/null | tail -1

In [ ]:
# Cell 2: Clone public repo + install Python deps
import os
if not os.path.exists('/content/DeepPEF/Megascale-fineTuning/pnas_train.py'):
    !rm -rf /content/DeepPEF
    !git clone https://github.com/nissimbrami/DeepEF-Thesis.git /content/DeepPEF
os.chdir('/content/DeepPEF')
print('cwd:', os.getcwd())

import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '')
print(f'torch={TORCH_VERSION}, cuda={CUDA_VERSION}, gpu={torch.cuda.get_device_name(0)}')

!pip install -q torch-geometric scipy scikit-learn pandas tqdm biopython wandb huggingface_hub
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION[:3]}.html 2>/dev/null || true

from torch_geometric.nn import GCNConv, GATv2Conv
print('PyG ok')

In [ ]:
# Cell 3: Download data from HuggingFace to Colab VM disk (no Drive used for data)
import os, shutil, glob
os.chdir('/content/DeepPEF')
from huggingface_hub import snapshot_download

LOCAL_DATA = '/content/hf_data'
td_check = f'{LOCAL_DATA}/training_data'
if os.path.isdir(td_check) and len(os.listdir(td_check)) > 300:
    print(f'Already downloaded: {len(os.listdir(td_check))} proteins')
else:
    print('Downloading nissimb/deepef-megascale (~50 GB)...')
    snapshot_download(repo_id='nissimb/deepef-megascale', repo_type='dataset',
                      local_dir=LOCAL_DATA, max_workers=8)
    print('Download done.')

for p in ['./data/MsDs/training_data', './data/MsDs/mutation_files',
          './data/ThermoMPNN', './data/Processed_K50_dG_datasets/Pnas_filtering']:
    if os.path.islink(p): os.unlink(p)
    elif os.path.isdir(p): shutil.rmtree(p)
os.makedirs('./data/MsDs', exist_ok=True)
os.makedirs('./data/ThermoMPNN', exist_ok=True)
os.makedirs('./data/Processed_K50_dG_datasets', exist_ok=True)

def find_in(name):
    direct = os.path.join(LOCAL_DATA, name)
    if os.path.exists(direct): return direct
    hits = glob.glob(f'{LOCAL_DATA}/**/{name}', recursive=True)
    return hits[0] if hits else None

def link_dir(src, dst):
    if src and os.path.isdir(src):
        os.symlink(src, dst); print(f'linked {dst} -> {src}')
    else:
        print(f'MISSING dir for {dst}')

def copy_if(src, dst):
    if src and os.path.isfile(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst); print(f'copied {os.path.basename(src)}')

link_dir(find_in('training_data'),  './data/MsDs/training_data')
link_dir(find_in('mutation_files'), './data/MsDs/mutation_files')
copy_if(find_in('mega_test.csv'),      './data/ThermoMPNN/mega_test.csv')
copy_if(find_in('mega_train.csv'),     './data/ThermoMPNN/mega_train.csv')
copy_if(find_in('train_proteins.csv'), './data/Processed_K50_dG_datasets/Pnas_filtering/train_proteins.csv')
copy_if(find_in('pnas_mutations.csv'), './data/Processed_K50_dG_datasets/Pnas_filtering/pnas_mutations.csv')

for p in ['./data/MsDs/mutation_files/.cache.csv', './data/MsDs/training_data/.cache']:
    if os.path.isfile(p): os.remove(p)
    elif os.path.isdir(p): shutil.rmtree(p)
for p in glob.glob('./**/__pycache__', recursive=True): shutil.rmtree(p)

proteins = [d for d in os.listdir('./data/MsDs/training_data')
            if os.path.isdir(os.path.join('./data/MsDs/training_data', d))]
print(f'\nProteins visible: {len(proteins)}')
assert len(proteins) >= 300, 'too few proteins - HF download incomplete?'

In [ ]:
# Cell 4: Evolution helpers - per-epoch logging, error capture, fallback
import os, re, json, shutil, time
import numpy as np
os.chdir('/content/DeepPEF')
os.environ['WANDB_MODE'] = 'disabled'
os.environ['PYTHONUNBUFFERED'] = '1'
os.makedirs('logs', exist_ok=True)

STATE_FILE = f'{OUT_ROOT}/results/state.json'

def load_state():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE) as f: return json.load(f)
    return {'current_best_pcc': 0.0, 'current_best_rung': None,
            'history': [], 'seeds': []}

def save_state(s):
    with open(STATE_FILE, 'w') as f: json.dump(s, f, indent=2)

def extract_pcc(log):
    if not os.path.exists(log): return None
    txt = open(log).read()
    for pat in [r'Best Pearson Correlation:\s*([0-9.]+)',
                r'Best Val PCC\s*=\s*([0-9.]+)',
                r'Best validation PCC:\s*([0-9.]+)',
                r'Final result: PCC\s*=\s*([0-9.]+)']:
        m = re.findall(pat, txt)
        if m: return float(m[-1])
    per_epoch = re.findall(r'Pearson Correlation:\s*([0-9.]+)', txt)
    if per_epoch: return max(float(x) for x in per_epoch)
    return None

def extract_per_epoch(log):
    if not os.path.exists(log): return []
    txt = open(log).read()
    out = []
    pat = re.compile(r'Epoch\s+(\d+)[\s\S]{0,2000}?Pearson Correlation:\s*([0-9.]+)',
                     re.IGNORECASE)
    for m in pat.finditer(txt):
        out.append([int(m.group(1)), float(m.group(2))])
    if out: return out
    pccs = re.findall(r'Pearson Correlation:\s*([0-9.]+)', txt)
    return [[i, float(v)] for i, v in enumerate(pccs)]

def extract_error(log):
    if not os.path.exists(log): return None
    txt = open(log).read()
    tb = re.findall(r'Traceback \(most recent call last\):[\s\S]+?(?=\n\S|\Z)', txt)
    if tb: return tb[-1].strip()[-2000:]
    errs = re.findall(
        r'^.*(?:RuntimeError|ValueError|KeyError|FileNotFoundError|AssertionError|OSError|CUDA out of memory).*$',
        txt, flags=re.MULTILINE)
    if errs: return errs[-1].strip()
    return None

def save_to_drive(rung_name, model_glob):
    os.system(f'cp logs/{rung_name}*.log {OUT_ROOT}/logs/ 2>/dev/null')
    tar = f'{OUT_ROOT}/models/{rung_name}.tar.gz'
    os.system(f'tar czf {tar} Megascale-fineTuning/models/{model_glob} 2>/dev/null')
    print(f'saved logs+models for {rung_name} -> {OUT_ROOT}')

def record_seed(rung_name, seed, log_file):
    pcc = extract_pcc(log_file)
    per_epoch = extract_per_epoch(log_file)
    err = extract_error(log_file)
    s = load_state()
    s.setdefault('seeds', []).append({
        'rung': rung_name, 'seed': seed,
        'best_pcc': pcc,
        'per_epoch': per_epoch,
        'error': err,
        'log': log_file,
        'time': time.strftime('%Y-%m-%d %H:%M:%S'),
    })
    save_state(s)
    if per_epoch:
        print('  per-epoch: ' + ' '.join(f'e{e}={v:.3f}' for e, v in per_epoch))
    if err:
        print(f'  ERROR: {err.splitlines()[-1][:200]}')
    return pcc

def update_best(rung_name, pccs):
    s = load_state()
    mean = float(np.mean(pccs)) if pccs else None
    prev_best = s['current_best_pcc']
    prev_rung = s['current_best_rung']
    entry = {'rung': rung_name, 'pccs': pccs, 'mean': mean,
             'time': time.strftime('%Y-%m-%d %H:%M:%S')}
    if mean is None:
        errs = [x['error'] for x in s.get('seeds', [])
                if x['rung'] == rung_name and x.get('error')]
        err_summary = errs[-1].splitlines()[-1][:300] if errs else 'no PCC parsed'
        verdict = f'FAILED -- {err_summary}'
        entry['error'] = err_summary
    elif mean > prev_best + 1e-4:
        s['current_best_pcc'] = mean
        s['current_best_rung'] = rung_name
        verdict = f'NEW BEST ({mean:.4f}, prev {prev_best:.4f})'
    elif prev_best - mean < 0.005:
        verdict = f'TIE -- keep prev best {prev_rung} ({prev_best:.4f})'
    else:
        delta = prev_best - mean
        verdict = f'FALLBACK -- kept {prev_rung} ({prev_best:.4f}); this rung lost by {delta:.4f}'
    entry['verdict'] = verdict
    s['history'].append(entry)
    save_state(s)
    print(f'\n[{rung_name}] mean PCC = {mean} | {verdict}')
    return mean, verdict

state = load_state()
print('current state:', state)

## Rung 0 - single-seed baseline sanity check (~40 min)

In [ ]:
%%time
import os; os.chdir('/content/DeepPEF')
RUNG = 'rung0_baseline'
SEED = 42
LOG = f'logs/{RUNG}_seed{SEED}.log'
!cd /content/DeepPEF && python -u Megascale-fineTuning/pnas_train.py \
    --model_name {RUNG}_seed{SEED} --seed {SEED} --dataset_type pnas \
    --epochs 15 --no_pretrained \
    --loss_type huber_rank --ranking_weight 0.1 \
    --use_knn_gat --one_mut --dg_ml \
    --cosine_lr --lr_min 1e-6 --weight_decay 1e-5 \
    --emb_type prott5 --mini_batch_size 16 --emb_projection none \
    2>&1 | tee {LOG}
pcc = record_seed(RUNG, SEED, LOG)
save_to_drive(RUNG, f'{RUNG}_*')
update_best(RUNG, [pcc] if pcc else [])

**Gate:** if Rung 0 < 0.48 -> stop and inspect `state.json` on Drive (it contains the error). Otherwise continue.

## Rung 1 - 5-seed ensemble of baseline (~3 hr)

In [ ]:
%%time
import os; os.chdir('/content/DeepPEF')
RUNG = 'rung1_baseline_5seed'
pccs = []
p0 = extract_pcc('logs/rung0_baseline_seed42.log')
if p0 is not None: pccs.append(p0)

for SEED in [123, 456, 789, 1337]:
    LOG = f'logs/{RUNG}_seed{SEED}.log'
    !cd /content/DeepPEF && python -u Megascale-fineTuning/pnas_train.py \
        --model_name {RUNG}_seed{SEED} --seed {SEED} --dataset_type pnas \
        --epochs 15 --no_pretrained \
        --loss_type huber_rank --ranking_weight 0.1 \
        --use_knn_gat --one_mut --dg_ml \
        --cosine_lr --lr_min 1e-6 --weight_decay 1e-5 \
        --emb_type prott5 --mini_batch_size 16 --emb_projection none \
        2>&1 | tee {LOG}
    p = record_seed(RUNG, SEED, LOG)
    if p is not None: pccs.append(p)
    print(f'seed {SEED}: PCC = {p}  | running list: {pccs}')
    save_to_drive(RUNG, f'{RUNG}_*')

update_best(RUNG, pccs)

## Rung 2 - + serial_fusion (~3 hr)

In [ ]:
%%time
import os; os.chdir('/content/DeepPEF')
RUNG = 'rung2_serial_fusion_5seed'
pccs = []
for SEED in [42, 123, 456, 789, 1337]:
    LOG = f'logs/{RUNG}_seed{SEED}.log'
    !cd /content/DeepPEF && python -u Megascale-fineTuning/pnas_train.py \
        --model_name {RUNG}_seed{SEED} --seed {SEED} --dataset_type pnas \
        --epochs 15 --no_pretrained \
        --loss_type huber_rank --ranking_weight 0.1 \
        --use_knn_gat --one_mut --dg_ml \
        --cosine_lr --lr_min 1e-6 --weight_decay 1e-5 \
        --emb_type prott5 --mini_batch_size 16 --emb_projection none \
        --serial_fusion --serial_fusion_dim 64 \
        2>&1 | tee {LOG}
    p = record_seed(RUNG, SEED, LOG)
    if p is not None: pccs.append(p)
    print(f'seed {SEED}: PCC = {p}  | running list: {pccs}')
    save_to_drive(RUNG, f'{RUNG}_*')

update_best(RUNG, pccs)

## Rung 3 - + use_learned_aa (stacks on rung 2 if it won; ~3 hr)

In [ ]:
%%time
import os; os.chdir('/content/DeepPEF')
RUNG = 'rung3_learned_aa_5seed'
state = load_state()
use_sf = state.get('current_best_rung') == 'rung2_serial_fusion_5seed'
extra = '--serial_fusion --serial_fusion_dim 64 ' if use_sf else ''
print(f'building on best={state["current_best_rung"]} | serial_fusion={use_sf}')

pccs = []
for SEED in [42, 123, 456, 789, 1337]:
    LOG = f'logs/{RUNG}_seed{SEED}.log'
    !cd /content/DeepPEF && python -u Megascale-fineTuning/pnas_train.py \
        --model_name {RUNG}_seed{SEED} --seed {SEED} --dataset_type pnas \
        --epochs 15 --no_pretrained \
        --loss_type huber_rank --ranking_weight 0.1 \
        --use_knn_gat --one_mut --dg_ml \
        --cosine_lr --lr_min 1e-6 --weight_decay 1e-5 \
        --emb_type prott5 --mini_batch_size 16 --emb_projection none \
        {extra} --use_learned_aa --aa_emb_dim 64 \
        2>&1 | tee {LOG}
    p = record_seed(RUNG, SEED, LOG)
    if p is not None: pccs.append(p)
    print(f'seed {SEED}: PCC = {p}  | running list: {pccs}')
    save_to_drive(RUNG, f'{RUNG}_*')

update_best(RUNG, pccs)

## Rung 4 - combine winners (only if rung 2 or 3 won; ~3 hr)

In [ ]:
%%time
import os; os.chdir('/content/DeepPEF')
state = load_state()
best = state['current_best_rung']
print(f'current best: {best} @ PCC {state["current_best_pcc"]:.4f}')

if best in ('rung2_serial_fusion_5seed', 'rung3_learned_aa_5seed'):
    RUNG = 'rung4_combined_5seed'
    pccs = []
    for SEED in [42, 123, 456, 789, 1337]:
        LOG = f'logs/{RUNG}_seed{SEED}.log'
        !cd /content/DeepPEF && python -u Megascale-fineTuning/pnas_train.py \
            --model_name {RUNG}_seed{SEED} --seed {SEED} --dataset_type pnas \
            --epochs 15 --no_pretrained \
            --loss_type huber_rank --ranking_weight 0.1 \
            --use_knn_gat --one_mut --dg_ml \
            --cosine_lr --lr_min 1e-6 --weight_decay 1e-5 \
            --emb_type prott5 --mini_batch_size 16 --emb_projection none \
            --serial_fusion --serial_fusion_dim 64 \
            --use_learned_aa --aa_emb_dim 64 \
            2>&1 | tee {LOG}
        p = record_seed(RUNG, SEED, LOG)
        if p is not None: pccs.append(p)
        print(f'seed {SEED}: PCC = {p}  | running list: {pccs}')
        save_to_drive(RUNG, f'{RUNG}_*')
    update_best(RUNG, pccs)
else:
    print('Skipping combined rung - neither rung 2 nor rung 3 won.')
    print('Current best:', best)

## Final summary + push everything to Drive

In [ ]:
import json
state = load_state()
print('='*70)
print(' EVOLUTION HISTORY')
print('='*70)
for h in state['history']:
    mean = f"{h['mean']:.4f}" if h['mean'] is not None else 'FAIL'
    print(f"  {h['rung']:<32} mean={mean}  {h['verdict']}")
print('='*70)
print(f"WINNER: {state['current_best_rung']} @ PCC {state['current_best_pcc']:.4f}")

print('\nPer-seed details (last 20):')
for x in state.get('seeds', [])[-20:]:
    pe = ' '.join(f"e{e}={v:.3f}" for e, v in (x.get('per_epoch') or []))
    err = f"  ERROR: {x['error'].splitlines()[-1][:160]}" if x.get('error') else ''
    print(f"  {x['rung']:<32} seed={x['seed']:<5} best={x['best_pcc']}  {pe}{err}")

!tar czf {OUT_ROOT}/results/all_logs.tar.gz logs/ 2>/dev/null && echo 'all_logs.tar.gz saved'
!cp {STATE_FILE} {OUT_ROOT}/results/state_final.json && echo 'state_final.json saved'
!ls -la {OUT_ROOT}/logs/ | head -20
!ls -la {OUT_ROOT}/models/